# Statistical Summary
Computes min, max, mean, median, and stddev for numeric columns.

**Configure the cell below**, then **Run All**. This notebook is read-only — no fix cell.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
# Columns to compute statistics for
NUMERIC_COLUMNS = {{NUMERIC_COLUMNS}}  # e.g., ["precio", "cantidad", "importe"]
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import percentile_approx
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()
df = spark.table(TABLE_NAME)
original_count = df.count()

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")

In [ ]:
print("=" * 60)
print("STATISTICAL SUMMARY — Numeric Columns")
print("=" * 60)

stats_results = []

for col_name in NUMERIC_COLUMNS:
    non_null_df = df.where(F.col(col_name).isNotNull())
    count_val = non_null_df.count()

    if count_val == 0:
        stats_results.append({
            "column": col_name, "count": 0,
            "min": None, "max": None, "mean": None, "median": None, "stddev": None
        })
        continue

    numeric_df = non_null_df.withColumn("_val", F.col(col_name).cast(DoubleType())).where(
        F.col("_val").isNotNull()
    )

    agg_result = numeric_df.agg(
        F.count("_val").alias("count"),
        F.min("_val").alias("min"),
        F.max("_val").alias("max"),
        F.mean("_val").alias("mean"),
        F.stddev("_val").alias("stddev"),
        percentile_approx("_val", 0.5).alias("median")
    ).collect()[0]

    stats_results.append({
        "column": col_name,
        "count": agg_result["count"],
        "min": round(agg_result["min"], 4) if agg_result["min"] is not None else None,
        "max": round(agg_result["max"], 4) if agg_result["max"] is not None else None,
        "mean": round(agg_result["mean"], 4) if agg_result["mean"] is not None else None,
        "median": round(agg_result["median"], 4) if agg_result["median"] is not None else None,
        "stddev": round(agg_result["stddev"], 4) if agg_result["stddev"] is not None else None,
    })

stats_df = spark.createDataFrame(stats_results)
display(stats_df)